## Combined Experiment 

Involving all the clauses

In [1]:
import pandas as pd
from torch.utils.data import DataLoader
from pathlib import Path
import evaluate
from openai import OpenAI
from datasets import Dataset, DatasetDict
from transformers import AdamW
from tqdm.auto import tqdm
from transformers import get_scheduler
from torch.nn.functional import softmax
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoTokenizer, DataCollatorWithPadding
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tools.utils import generate_response
from tools.prompt_templates import generate_negative_prompts_few_shot, generate_positive_prompts_few_shot
import pandas as pd
import re

In [2]:
import torch
# answerdotai/ModernBERT-base distilbert-base-uncased

def train_model(clause_type,train,test,prompt_type,checkpoint= "bert-base-uncased", num_epochs = 10):
    def tokenize_function(example):
        return tokenizer(example["text"], truncation=True)
    
    scores_all = []

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available() else "cpu"
        )

    torch.manual_seed(1984)

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=4)
    model.to(device)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    # Combine into a DatasetDict
    dataset = DatasetDict({
        'train':  Dataset.from_pandas(train),
        'test': Dataset.from_pandas(test[['text','labels']].reset_index(drop=True)),
    })

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    tokenized_datasets = tokenized_datasets.remove_columns(["text"])
    tokenized_datasets.set_format("torch")

    train_dataloader = DataLoader(
        tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
            )
    eval_dataloader = DataLoader(
        tokenized_datasets["test"], batch_size=8, collate_fn=data_collator
        )
    
    optimizer = AdamW(model.parameters(), lr=1e-5, eps=1e-6, weight_decay=0.2)

    
    num_training_steps = num_epochs * len(train_dataloader)

    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=2,
        num_training_steps=num_training_steps,
    )

    progress_bar = tqdm(range(num_training_steps))
    #metric = evaluate.load("glue", "mrpc")
    metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")
    

    model.train()
    best_score = .0
    for epoch in range(num_epochs):
        for batch in train_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)


        model.eval()
        for batch in eval_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.no_grad():
                outputs = model(**batch)

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            metric.add_batch(predictions=predictions, references=batch["labels"])
            f1_metric.add_batch(predictions=predictions, references=batch["labels"])

        scores = metric.compute()
        scores['f1'] = f1_metric.compute(average="macro")['f1']
        #print(scores, f1_scores)
        scores_all.append(scores)
        print(f"Epoch {epoch}:", scores)

        if scores["f1"] > best_score:
            print("Saving model")
            best_score = scores["f1"]
            model.save_pretrained(f"./models/combined_{prompt_type}_model")
            tokenizer.save_pretrained(f"./models/combined_{prompt_type}_model")
    return scores_all, best_score
   


In [3]:

processed_data_dir = Path('processed_data/multigenre')
metadata = pd.read_csv(processed_data_dir / 'metadata.tsv',sep='\t')
metadata.shape

(226872, 6)

In [4]:
annotator = '_TS'
train_dfs, test_dfs = [], []


for i,clause_type in enumerate(['arbitration', 'opt-out', 'class waiver']):
    annotations_df = pd.read_csv(f'annotations/manual/{clause_type}_annotations_gpt4{annotator}.csv', index_col=0)
    annotations_df = annotations_df[~annotations_df.labels.isnull()]
    annotations_df.labels = annotations_df.labels.astype(int)
    annotations_df.replace({'labels': {2: i+1, 1:i+1}}, inplace=True)
    
    annotations_df.labels.value_counts(normalize=True)

    annotations_df = annotations_df.merge(metadata, left_on='text', right_on='sentence_modified', how='left')
    annotations_df = annotations_df[['sentence_original','labels']]
    annotations_df.columns = ['text','labels']

    annotations_df.drop_duplicates(subset='text', inplace=True)
    annotations_df['text'] = annotations_df['text'].apply(lambda x: re.sub(r'\n', ' ', x))

    train,test = train_test_split(annotations_df, test_size=0.75, random_state=42)
    print('train size',train.labels.value_counts())
    print('test size',test.labels.value_counts())
    train_dfs.append(train)
    test_dfs.append(test)

train = pd.concat(train_dfs, axis=0, ignore_index=True)
test = pd.concat(test_dfs, axis=0, ignore_index=True)
print('train size',train.labels.value_counts())

train size labels
1    19
0     6
Name: count, dtype: int64
test size labels
1    64
0    11
Name: count, dtype: int64
train size labels
0    17
2     9
Name: count, dtype: int64
test size labels
2    41
0    37
Name: count, dtype: int64
train size labels
3    17
0     9
Name: count, dtype: int64
test size labels
3    52
0    29
Name: count, dtype: int64
train size labels
0    32
1    19
3    17
2     9
Name: count, dtype: int64


In [5]:
df_sample_train = metadata.sample(n=200, random_state=0).reset_index(drop=True)
train_sents = [s for s in df_sample_train.sentence_original.to_list() if s not in train.text.to_list()]
df_sample_train = pd.DataFrame(train_sents, columns=['text'])
df_sample_train['labels'] = 0
train_data = pd.concat([train[['text','labels']], df_sample_train[['text','labels']] ], axis=0, ignore_index=True).reset_index(drop=True)

df_sample_test = metadata.sample(n=50, random_state=2).reset_index(drop=True)
test_sents = [s for s in df_sample_test.sentence_original.to_list() if s not in train_data.text.to_list()]
df_sample_test = pd.DataFrame(train_sents, columns=['text'])
df_sample_test['labels'] = 0
test_data = pd.concat([test[['text','labels']], df_sample_test[['text','labels']] ], axis=0, ignore_index=True).reset_index(drop=True)
train_data.shape, test_data.shape

((276, 2), (433, 2))

In [6]:
syntethic_data_all = []
for i,clause_type in enumerate(['arbitration', 'opt-out', 'class waiver']):
    syntethic_data = pd.read_csv(f'annotations/synthetic/{clause_type}_synthetic_gpt4.csv', index_col=0)
    syntethic_data.task.replace({'negative_prompts_zeo_shot': 'negative_prompts_zero_shot'}, inplace=True)
    syntethic_data['task'] = syntethic_data.task.apply(lambda x: '_'.join(x.split('_')[2:]))
    
    syntethic_data.replace({'labels': {1:i+1}}, inplace=True)
    syntethic_data_all.append(syntethic_data)

syntethic_data_all = pd.concat(syntethic_data_all, axis=0, ignore_index=True) 
syntethic_data_all.value_counts('labels')

labels
0    900
1    300
2    300
3    300
Name: count, dtype: int64

In [ ]:

results = {}
results['train_set_only'] = train_model(clause_type,train_data,test_data,'train_set_only')

for prompt_type in syntethic_data_all.task.unique():
    # add synthetic data
    if prompt_type.startswith('positive'):
        syntethic_data = syntethic_data_all[syntethic_data_all.task==prompt_type]
        train_data_syn = pd.concat([train_data, syntethic_data[['text','labels']]], axis=0, ignore_index=True).reset_index(drop=True)
        results[prompt_type] = train_model(clause_type,train_data_syn,test_data,prompt_type)

/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/276 [00:00<?, ? examples/s]

Map:   0%|          | 0/433 [00:00<?, ? examples/s]

/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  0%|          | 0/350 [00:00<?, ?it/s]

...

Map:   0%|          | 0/876 [00:00<?, ? examples/s]

Map:   0%|          | 0/433 [00:00<?, ? examples/s]

/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  0%|          | 0/1100 [00:00<?, ?it/s]

Epoch 0: {'accuracy': 0.745958429561201, 'f1': 0.5041354323543291}
Saving model
Epoch 1: {'accuracy': 0.7575057736720554, 'f1': 0.6218760719133309}
Saving model
Epoch 2: {'accuracy': 0.7713625866050808, 'f1': 0.6091711566240736}
Epoch 3: {'accuracy': 0.7875288683602771, 'f1': 0.6499648576080524}
Saving model
Epoch 4: {'accuracy': 0.789838337182448, 'f1': 0.6628833334532342}
Saving model
Epoch 5: {'accuracy': 0.7852193995381063, 'f1': 0.6576105732231005}
Epoch 6: {'accuracy': 0.7806004618937644, 'f1': 0.6448919908466819}
Epoch 7: {'accuracy': 0.7759815242494227, 'f1': 0.6406872216000995}
Epoch 8: {'accuracy': 0.7806004618937644, 'f1': 0.6519243511076542}
Epoch 9: {'accuracy': 0.7806004618937644, 'f1': 0.6519243511076542}


In [8]:
results

{'train_set_only': ([{'accuracy': 0.6374133949191686,
    'f1': 0.19464033850493653},
   {'accuracy': 0.6374133949191686, 'f1': 0.19464033850493653},
   {'accuracy': 0.6374133949191686, 'f1': 0.19464033850493653},
   {'accuracy': 0.6374133949191686, 'f1': 0.19464033850493653},
   {'accuracy': 0.6997690531177829, 'f1': 0.3768643253640873},
   {'accuracy': 0.7159353348729792, 'f1': 0.4371770367333475},
   {'accuracy': 0.7251732101616628, 'f1': 0.46745261984392417},
   {'accuracy': 0.7390300230946882, 'f1': 0.5039966739202912},
   {'accuracy': 0.74364896073903, 'f1': 0.5111333444268796},
   {'accuracy': 0.7413394919168591, 'f1': 0.5077370107962214}],
  0.5111333444268796),
 'zero_shot': ([{'accuracy': 0.6812933025404158, 'f1': 0.3352806955427043},
   {'accuracy': 0.7367205542725174, 'f1': 0.4646930240164717},
   {'accuracy': 0.7875288683602771, 'f1': 0.7088318012468691},
   {'accuracy': 0.7528868360277137, 'f1': 0.5897904781275309},
   {'accuracy': 0.7505773672055427, 'f1': 0.592097955236

In [11]:
table = []
for prompt,scores in results.items():
        table.append([ prompt,max([s['f1'] for s in scores[0]]),max([s['accuracy'] for s in scores[0]])])

df_res = pd.DataFrame(table, columns=['prompt','f1','accuracy'])
df_res.to_csv(f'results/results_combined_model.csv', index=False)
       

In [12]:
df_res

,prompt,f1,accuracy
0,train_set_only,0.511133,0.743649
1,zero_shot,0.708832,0.787529
2,few_shot,0.704887,0.782910
3,contrastive_few_shot,0.742636,0.799076
